In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
import lightgbm as lgb

In [4]:
import pandas as pd

# ===== CELL 2: Load and Prepare Data =====
print("\n📊 LOADING DATA...")
DATA_DIR = "/kaggle/input/playground-series-s6e1"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")


📊 LOADING DATA...


Split features and target

In [5]:
y = train["exam_score"]

X = train.drop(columns=["exam_score", "id"])
X_test = test.drop(columns=["id"])

Detect categorical features (SAFE)

In [6]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
cat_features_idx = [X.columns.get_loc(col) for col in cat_features]

5-Fold Cross-Validation (CatBoost)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
cb_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"CatBoost Fold {fold + 1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        loss_function="RMSE",
        random_seed=42 + fold,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150
    )

    oof_preds[val_idx] = model.predict(X_val)
    cb_test_preds += model.predict(X_test) / kf.n_splits

CatBoost Fold 1


CV Score (sanity check)

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = mean_squared_error(y, oof_preds) ** 0.5
print("CatBoost CV RMSE:", rmse)

Train LightGBM (second perspective)

In [ ]:
lgb_train = lgb.Dataset(
    X,
    y,
    categorical_feature=cat_features,
    free_raw_data=False
)

lgb_model = lgb.train(
    {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 64,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "seed": 42
    },
    lgb_train,
    num_boost_round=2500
)

lgb_test_preds = lgb_model.predict(X_test)

Blend predictions (THIS IS THE JUMP),This reduces variance from synthetic artifacts.

In [ ]:
final_preds = 0.6 * cb_test_preds + 0.4 * lgb_test_preds

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "exam_score": final_preds
})

submission.to_csv("submission.csv", index=False)